In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
print("all library imported successfully")

all library imported successfully


In [17]:
from pathlib import Path

# Define dataset folder path
data_path = Path("../data/raw")

# Load all datasets
products = pd.read_csv(data_path / "Products.csv")
customers = pd.read_csv(data_path / "Customers.csv")
suppliers = pd.read_csv(data_path / "Suppliers.csv")
sales = pd.read_csv(data_path / "Sales.csv")
inventory = pd.read_csv(data_path / "Inventory.csv")
purchase = pd.read_csv(data_path / "Purchase.csv")
payments = pd.read_csv(data_path / "Payments.csv")

print("All 7 datasets loaded successfully!")

All 7 datasets loaded successfully!


In [18]:
datasets = {
    "Products": products,
    "Customers": customers,
    "Suppliers": suppliers,
    "Sales": sales,
    "Inventory": inventory,
    "Purchase": purchase,
    "Payments": payments
}

for name, df in datasets.items():
    print(name, ":", df.shape)

Products : (500, 9)
Customers : (2000, 5)
Suppliers : (10, 5)
Sales : (45225, 8)
Inventory : (6018, 6)
Purchase : (3026, 8)
Payments : (6330, 5)


In [19]:
for name, df in datasets.items():
    print("\nDataset:", name)
    print("-" * 50)
    print(df.info())


Dataset: Products
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ProductID     500 non-null    object 
 1   ProductName   500 non-null    object 
 2   Category      492 non-null    object 
 3   Brand         500 non-null    object 
 4   Price         494 non-null    float64
 5   CostPrice     500 non-null    float64
 6   SupplierID    500 non-null    object 
 7   ReorderLevel  500 non-null    int64  
 8   LeadTimeDays  500 non-null    int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 35.3+ KB
None

Dataset: Customers
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   CustomerID   

In [20]:
for name, df in datasets.items():
    print("\nDataset:", name)
    print(df.isnull().sum())
    print("Total missing values:", df.isnull().sum().sum())


Dataset: Products
ProductID       0
ProductName     0
Category        8
Brand           0
Price           6
CostPrice       0
SupplierID      0
ReorderLevel    0
LeadTimeDays    0
dtype: int64
Total missing values: 14

Dataset: Customers
CustomerID         0
CustomerName       0
City               0
CustomerSegment    0
SignupDate         0
dtype: int64
Total missing values: 0

Dataset: Suppliers
SupplierID             0
SupplierName           0
Location               0
AverageLeadTimeDays    0
DelayRate              0
dtype: int64
Total missing values: 0

Dataset: Sales
SalesID           0
InvoiceID         0
ProductID         0
CustomerID       60
Quantity          0
UnitPrice        60
SalesDate        60
PaymentMethod     0
dtype: int64
Total missing values: 180

Dataset: Inventory
InventoryID          0
ProductID            0
AvailableStock       0
WarehouseID         25
SnapshotDate         0
ExpiryDate        5416
dtype: int64
Total missing values: 5441

Dataset: Purchase
Purch

In [21]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_summary.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": df[column].isnull().sum(),
            "Missing_Percentage": round(
                df[column].isnull().mean() * 100, 2
            )
        })

missing_df = pd.DataFrame(missing_summary)

missing_df = missing_df[missing_df["Missing_Count"] > 0]

print(missing_df.to_string(index=False))

  Dataset       Column  Missing_Count  Missing_Percentage
 Products     Category              8                1.60
 Products        Price              6                1.20
    Sales   CustomerID             60                0.13
    Sales    UnitPrice             60                0.13
    Sales    SalesDate             60                0.13
Inventory  WarehouseID             25                0.42
Inventory   ExpiryDate           5416               90.00
 Purchase DeliveryDate             25                0.83
 Payments  PaymentDate           1923               30.38


In [22]:
# Products: Missing Category and Price may affect product classification and revenue calculations.



# Sales: Missing CustomerID affects customer-level analysis. Missing SalesDate and UnitPrice can affect time-based and revenue analysis.



# Inventory: WarehouseID is missing in 25 records. ExpiryDate is missing in 90% of records, which may be expected for products that don't expire.



# Purchase: Missing DeliveryDate may affect supplier delivery-performance calculations.



# Payments: Missing PaymentDate needs investigation before cleaning.

In [23]:
for name, df in datasets.items():

    print("\nDataset:", name)
    print("Total duplicate rows:", df.duplicated().sum())


Dataset: Products
Total duplicate rows: 0

Dataset: Customers
Total duplicate rows: 0

Dataset: Suppliers
Total duplicate rows: 0

Dataset: Sales
Total duplicate rows: 222

Dataset: Inventory
Total duplicate rows: 18

Dataset: Purchase
Total duplicate rows: 14

Dataset: Payments
Total duplicate rows: 0


In [24]:
duplicate_checks = {
    "Products": (products, "ProductID"),
    "Customers": (customers, "CustomerID"),
    "Suppliers": (suppliers, "SupplierID"),
    "Sales": (sales, "InvoiceID"),
    "Inventory": (inventory, ["ProductID", "SnapshotDate"]),
    "Purchase": (purchase, "PurchaseID"),
    "Payments": (payments, "PaymentID")
}

for name, (df, key) in duplicate_checks.items():

    print("\nDataset:", name)
    print("Duplicate business keys:", df.duplicated(subset=key).sum())


Dataset: Products
Duplicate business keys: 0

Dataset: Customers
Duplicate business keys: 0

Dataset: Suppliers
Duplicate business keys: 0

Dataset: Sales
Duplicate business keys: 225

Dataset: Inventory
Duplicate business keys: 18

Dataset: Purchase
Duplicate business keys: 15

Dataset: Payments
Duplicate business keys: 0


In [25]:
print("Purchase columns:")
print(purchase.columns.tolist())

print("\nInventory columns:")
print(inventory.columns.tolist())

Purchase columns:
['PurchaseID', 'SupplierID', 'ProductID', 'Quantity', 'OrderDate', 'DeliveryDate', 'PurchaseCost', 'PurchaseStatus']

Inventory columns:
['InventoryID', 'ProductID', 'AvailableStock', 'WarehouseID', 'SnapshotDate', 'ExpiryDate']


In [26]:
checks = {
    "Sales": (sales, ["Quantity", "UnitPrice"]),
    "Purchase": (purchase, ["Quantity", "PurchaseCost"]),
    "Inventory": (inventory, ["AvailableStock"])
}

for name, (df, columns) in checks.items():

    print("\nDataset:", name)

    for column in columns:
        print(column, "Negative values:", (df[column] < 0).sum())
        print(column, "Zero values:", (df[column] == 0).sum())


Dataset: Sales
Quantity Negative values: 45
Quantity Zero values: 0
UnitPrice Negative values: 0
UnitPrice Zero values: 0

Dataset: Purchase
Quantity Negative values: 15
Quantity Zero values: 0
PurchaseCost Negative values: 0
PurchaseCost Zero values: 0

Dataset: Inventory
AvailableStock Negative values: 20
AvailableStock Zero values: 362


In [27]:
#Check date ranges and future dates

date_columns = {
    "Sales": (sales, "SalesDate"),
    "Purchase": (purchase, "OrderDate"),
    "Inventory": (inventory, "SnapshotDate"),
    "Payments": (payments, "PaymentDate")
}

for name, (df, column) in date_columns.items():

    dates = pd.to_datetime(
        df[column],
        errors="coerce",
        dayfirst=True
    )

    print("\nDataset:", name)
    print("Minimum date:", dates.min())
    print("Maximum date:", dates.max())
    print("Missing in original data:", df[column].isna().sum())
    print("Invalid date values:", (dates.isna() & df[column].notna()).sum())


Dataset: Sales
Minimum date: 2025-10-31 00:00:00
Maximum date: 2026-11-05 00:00:00
Missing in original data: 60
Invalid date values: 0

Dataset: Purchase
Minimum date: 2025-10-31 00:00:00
Maximum date: 2026-10-30 00:00:00
Missing in original data: 0
Invalid date values: 0

Dataset: Inventory
Minimum date: 2025-11-30 00:00:00
Maximum date: 2026-10-30 00:00:00
Missing in original data: 0
Invalid date values: 0

Dataset: Payments
Minimum date: 2025-11-02 00:00:00
Maximum date: 2026-11-27 00:00:00
Missing in original data: 1923
Invalid date values: 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_15752\1474784036.py:12: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dates = pd.to_datetime(


In [28]:

#Check referential integrity
checks = {
    "Sales -> Products": (sales, "ProductID", products, "ProductID"),
    "Sales -> Customers": (sales, "CustomerID", customers, "CustomerID"),
    "Purchase -> Products": (purchase, "ProductID", products, "ProductID"),
    "Purchase -> Suppliers": (purchase, "SupplierID", suppliers, "SupplierID"),
    "Inventory -> Products": (inventory, "ProductID", products, "ProductID")
}

for name, (child, child_col, parent, parent_col) in checks.items():

    unmatched = child[
        child[child_col].notna() &
        ~child[child_col].isin(parent[parent_col])
    ]

    missing = child[child_col].isna().sum()

    print("\n", name)
    print("Missing IDs:", missing)
    print("Unmatched IDs:", len(unmatched))


 Sales -> Products
Missing IDs: 0
Unmatched IDs: 0

 Sales -> Customers
Missing IDs: 60
Unmatched IDs: 0

 Purchase -> Products
Missing IDs: 0
Unmatched IDs: 0

 Purchase -> Suppliers
Missing IDs: 0
Unmatched IDs: 0

 Inventory -> Products
Missing IDs: 0
Unmatched IDs: 0


In [29]:
#Check remaining relationships

sales_invoices = set(sales["InvoiceID"].dropna())
payment_invoices = set(payments["InvoiceID"].dropna())

print("Payments without matching Sales Invoice:")
print(len(payment_invoices - sales_invoices))

print("\nSales invoices without payment records:")
print(len(sales_invoices - payment_invoices))

Payments without matching Sales Invoice:
0

Sales invoices without payment records:
38670


In [30]:

# PHASE 2 - CONSOLIDATED DATA QUALITY REPORT

from pathlib import Path
import pandas as pd

# Create report folder
report_path = Path("../reports/data_quality")
report_path.mkdir(parents=True, exist_ok=True)

# 1. DATASET OVERVIEW

overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Total_Columns": len(df.columns),
        "Duplicate_Rows": df.duplicated().sum(),
        "Total_Missing_Values": df.isnull().sum().sum()
    })

overview_df = pd.DataFrame(overview)


# 2. MISSING VALUE REPORT

missing_report = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        if missing_count > 0:
            missing_report.append({
                "Dataset": name,
                "Column": column,
                "Missing_Count": missing_count,
                "Missing_Percentage": round(
                    df[column].isnull().mean() * 100, 2
                )
            })

missing_report_df = pd.DataFrame(missing_report)


# 3. DUPLICATE REPORT

duplicate_keys = {
    "Products": ["ProductID"],
    "Customers": ["CustomerID"],
    "Suppliers": ["SupplierID"],
    "Sales": ["InvoiceID"],
    "Inventory": ["ProductID", "SnapshotDate"],
    "Purchase": ["PurchaseID"],
    "Payments": ["PaymentID"]
}

duplicate_report = []

for name, df in datasets.items():
    duplicate_report.append({
        "Dataset": name,
        "Complete_Duplicate_Rows": df.duplicated().sum(),
        "Business_Key_Duplicates": df.duplicated(
            subset=duplicate_keys[name]
        ).sum()
    })

duplicate_report_df = pd.DataFrame(duplicate_report)


# 4. NUMERICAL QUALITY REPORT

numeric_columns = {
    "Sales": ["Quantity", "UnitPrice"],
    "Purchase": ["Quantity", "PurchaseCost"],
    "Inventory": ["AvailableStock"]
}

numeric_report = []

for name, columns in numeric_columns.items():
    df = datasets[name]

    for column in columns:
        numeric_report.append({
            "Dataset": name,
            "Column": column,
            "Negative_Values": (df[column] < 0).sum(),
            "Zero_Values": (df[column] == 0).sum()
        })

numeric_report_df = pd.DataFrame(numeric_report)


# 5. DATE QUALITY REPORT

date_columns = {
    "Sales": (sales, "SalesDate"),
    "Purchase": (purchase, "OrderDate"),
    "Inventory": (inventory, "SnapshotDate"),
    "Payments": (payments, "PaymentDate")
}

reference_date = pd.Timestamp("2026-10-30")

date_report = []

for name, (df, column) in date_columns.items():

    dates = pd.to_datetime(
        df[column],
        errors="coerce",
        format="mixed",
        dayfirst=True
    )

    date_report.append({
        "Dataset": name,
        "Date_Column": column,
        "Minimum_Date": dates.min(),
        "Maximum_Date": dates.max(),
        "Missing_Dates": df[column].isna().sum(),
        "Invalid_Dates": (
            dates.isna() & df[column].notna()
        ).sum(),
        "Future_Dates_After_2026_10_30": (
            dates > reference_date
        ).sum()
    })

date_report_df = pd.DataFrame(date_report)


# 6. REFERENTIAL INTEGRITY REPORT

relationships = [
    ("Sales", sales, "ProductID", products, "ProductID"),
    ("Sales", sales, "CustomerID", customers, "CustomerID"),
    ("Purchase", purchase, "ProductID", products, "ProductID"),
    ("Purchase", purchase, "SupplierID", suppliers, "SupplierID"),
    ("Inventory", inventory, "ProductID", products, "ProductID")
]

integrity_report = []

for name, child, child_col, parent, parent_col in relationships:

    missing_count = child[child_col].isna().sum()

    unmatched_count = (
        child[child_col].notna() &
        ~child[child_col].isin(parent[parent_col])
    ).sum()

    integrity_report.append({
        "Relationship": name + " -> " + parent_col,
        "Missing_IDs": missing_count,
        "Unmatched_IDs": unmatched_count
    })


# Sales and Payments relationship

sales_invoices = set(sales["InvoiceID"].dropna())
payment_invoices = set(payments["InvoiceID"].dropna())

integrity_report.append({
    "Relationship": "Payments -> Sales InvoiceID",
    "Missing_IDs": payments["InvoiceID"].isna().sum(),
    "Unmatched_IDs": len(payment_invoices - sales_invoices)
})

integrity_report.append({
    "Relationship": "Sales -> Payments InvoiceID",
    "Missing_IDs": sales["InvoiceID"].isna().sum(),
    "Unmatched_IDs": len(sales_invoices - payment_invoices)
})

integrity_report_df = pd.DataFrame(integrity_report)


# 7. SAVE ALL REPORTS

reports = {
    "Dataset_Overview": overview_df,
    "Missing_Values": missing_report_df,
    "Duplicate_Checks": duplicate_report_df,
    "Numerical_Quality": numeric_report_df,
    "Date_Quality": date_report_df,
    "Referential_Integrity": integrity_report_df
}

for report_name, report_df in reports.items():
    report_df.to_csv(
        report_path / (report_name + ".csv"),
        index=False
    )

print("PHASE 2 DATA QUALITY REPORT GENERATED SUCCESSFULLY!")

print("\nReports saved in:", report_path)

for report_name in reports:
    print("-", report_name + ".csv")


PHASE 2 DATA QUALITY REPORT GENERATED SUCCESSFULLY!

Reports saved in: ..\reports\data_quality
- Dataset_Overview.csv
- Missing_Values.csv
- Duplicate_Checks.csv
- Numerical_Quality.csv
- Date_Quality.csv
- Referential_Integrity.csv
